# Notebook 05v2: Full Evaluation — Table 1 (Bug-Fixed)

Changes from v1:
- All bugs fixed (enable_thinking, cosine_loss, prefix embedding)
- n=500 evaluation (paper scale)
- v2 checkpoints (AE, adapter)

Expected: Single Answer MATH ~45%, ThoughtComm > Debate

In [ ]:
import os, sys, subprocess
REPO = 'thoughtcomm'
if os.path.exists(REPO):
    os.chdir(REPO)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AUMEZAK/thoughtcomm.git'], check=True)
    os.chdir(REPO)
    subprocess.run(['pip', 'install', '-e', '.', '-q'], check=True)

In [ ]:
import torch
import os
from configs.config import ThoughtCommConfig
from models.model_utils import load_model_and_tokenizer
from models.autoencoder import SparsityRegularizedAE
from models.prefix_adapter import PrefixAdapter
from pipeline.agreement import AgreementReweighter
from pipeline.thought_comm import ThoughtCommPipeline, run_single_answer_baseline, run_debate_baseline
from data.math_data import load_math_dataset
from data.gsm8k_data import load_gsm8k_dataset
from evaluation.metrics import compute_accuracy, compute_consensus, compute_accuracy_with_std

from google.colab import drive
drive.mount('/content/drive')

config = ThoughtCommConfig.for_qwen_0_6b()
MODEL_TAG = config.model_name.replace('/', '_')
SAVE_DIR = config.save_dir

print(f'Model: {config.model_name}')
print(f'Eval size: {config.num_eval}')

In [ ]:
# Load model
model, tokenizer = load_model_and_tokenizer(
    config.model_name, dtype=config.torch_dtype
)

# Load eval datasets
_, math_eval = load_math_dataset(
    num_train=config.num_train, num_eval=config.num_eval, level=config.math_level
)
_, gsm8k_eval = load_gsm8k_dataset(
    num_train=config.num_train, num_eval=config.num_eval
)
print(f'MATH eval: {len(math_eval)}, GSM8K eval: {len(gsm8k_eval)}')

## 1. Single Answer Baseline

In [ ]:
# MATH - Single Answer
sa_math = run_single_answer_baseline(model, tokenizer, math_eval, config)
sa_math_acc, _ = compute_accuracy(sa_math, [e['answer'] for e in math_eval], 'math')
print(f'Single Answer MATH: {sa_math_acc:.1f}%')
print(f'  (Paper: 45.80% — if we get ~40-50%, enable_thinking fix is working)')

In [ ]:
# GSM8K - Single Answer
sa_gsm8k = run_single_answer_baseline(model, tokenizer, gsm8k_eval, config)
sa_gsm8k_acc, _ = compute_accuracy(sa_gsm8k, [e['answer'] for e in gsm8k_eval], 'gsm8k')
print(f'Single Answer GSM8K: {sa_gsm8k_acc:.1f}%')
print(f'  (Paper: 58.20%)')

## 2. Debate-Only Baseline

In [ ]:
# MATH - Debate
debate_math = run_debate_baseline(model, tokenizer, math_eval, config)
debate_math_acc_per_agent = []
for agent_idx in range(config.num_agents):
    agent_resp = [r[agent_idx] for r in debate_math['final_responses']]
    acc, _ = compute_accuracy(agent_resp, [e['answer'] for e in math_eval], 'math')
    debate_math_acc_per_agent.append(acc)
debate_math_acc = sum(debate_math_acc_per_agent) / len(debate_math_acc_per_agent)
debate_math_cons = compute_consensus(debate_math['final_responses'], [e['answer'] for e in math_eval], 'math')
print(f'Debate MATH: {debate_math_acc:.1f}% (agents: {debate_math_acc_per_agent}), Consensus: {debate_math_cons:.1f}%')

In [ ]:
# GSM8K - Debate
debate_gsm8k = run_debate_baseline(model, tokenizer, gsm8k_eval, config)
debate_gsm8k_acc_per_agent = []
for agent_idx in range(config.num_agents):
    agent_resp = [r[agent_idx] for r in debate_gsm8k['final_responses']]
    acc, _ = compute_accuracy(agent_resp, [e['answer'] for e in gsm8k_eval], 'gsm8k')
    debate_gsm8k_acc_per_agent.append(acc)
debate_gsm8k_acc = sum(debate_gsm8k_acc_per_agent) / len(debate_gsm8k_acc_per_agent)
debate_gsm8k_cons = compute_consensus(debate_gsm8k['final_responses'], [e['answer'] for e in gsm8k_eval], 'gsm8k')
print(f'Debate GSM8K: {debate_gsm8k_acc:.1f}% (agents: {debate_gsm8k_acc_per_agent}), Consensus: {debate_gsm8k_cons:.1f}%')

## 3. ThoughtComm (Ours)

In [ ]:
# Load v2 checkpoints
ae = SparsityRegularizedAE(config.n_h, config.n_z, config.ae_hidden, config.ae_num_layers)
ae_dir = os.path.join(SAVE_DIR, f'{MODEL_TAG}_ae_v2')
ae.load_state_dict(torch.load(os.path.join(ae_dir, 'ae.pt'), weights_only=True, map_location='cpu'), strict=False)

B = torch.load(os.path.join(ae_dir, 'B.pt'), weights_only=True, map_location='cpu')
reweighter = AgreementReweighter(B, config)

adapter_dir = os.path.join(SAVE_DIR, f'{MODEL_TAG}_adapter_v2')
reweighter.load_state_dict(
    torch.load(os.path.join(adapter_dir, 'reweighter.pt'), weights_only=True, map_location='cpu'),
    strict=False
)

adapter = PrefixAdapter(config.n_z, config.hidden_size, config.adapter_hidden, config.prefix_length)
adapter.load_state_dict(
    torch.load(os.path.join(adapter_dir, 'adapter.pt'), weights_only=True, map_location='cpu')
)

tc = ThoughtCommPipeline(model, tokenizer, ae, reweighter, adapter, config)
print('ThoughtComm pipeline loaded')

In [ ]:
# MATH - ThoughtComm
tc_math = tc.evaluate(math_eval, 'math')
tc_math_acc_per_agent = []
for agent_idx in range(config.num_agents):
    agent_resp = [r[agent_idx] for r in tc_math['final_responses']]
    acc, _ = compute_accuracy(agent_resp, tc_math['ground_truths'], 'math')
    tc_math_acc_per_agent.append(acc)
tc_math_acc = sum(tc_math_acc_per_agent) / len(tc_math_acc_per_agent)
tc_math_cons = compute_consensus(tc_math['final_responses'], tc_math['ground_truths'], 'math')
print(f'ThoughtComm MATH: {tc_math_acc:.1f}% (agents: {tc_math_acc_per_agent}), Consensus: {tc_math_cons:.1f}%')

In [ ]:
# GSM8K - ThoughtComm
tc_gsm8k = tc.evaluate(gsm8k_eval, 'gsm8k')
tc_gsm8k_acc_per_agent = []
for agent_idx in range(config.num_agents):
    agent_resp = [r[agent_idx] for r in tc_gsm8k['final_responses']]
    acc, _ = compute_accuracy(agent_resp, tc_gsm8k['ground_truths'], 'gsm8k')
    tc_gsm8k_acc_per_agent.append(acc)
tc_gsm8k_acc = sum(tc_gsm8k_acc_per_agent) / len(tc_gsm8k_acc_per_agent)
tc_gsm8k_cons = compute_consensus(tc_gsm8k['final_responses'], tc_gsm8k['ground_truths'], 'gsm8k')
print(f'ThoughtComm GSM8K: {tc_gsm8k_acc:.1f}% (agents: {tc_gsm8k_acc_per_agent}), Consensus: {tc_gsm8k_cons:.1f}%')

## 4. Results Summary (Table 1)

In [ ]:
print('=' * 85)
print(f'Table 1 Results (v2) — {config.model_name}')
print('=' * 85)
print(f'{"Method":<20} {"MATH Acc":>10} {"MATH Cons":>10} {"GSM8K Acc":>10} {"GSM8K Cons":>10}')
print('-' * 85)
print(f'{"Single Answer":<20} {sa_math_acc:>9.1f}% {"N/A":>10} {sa_gsm8k_acc:>9.1f}% {"N/A":>10}')
print(f'{"Debate Only":<20} {debate_math_acc:>9.1f}% {debate_math_cons:>9.1f}% {debate_gsm8k_acc:>9.1f}% {debate_gsm8k_cons:>9.1f}%')
print(f'{"ThoughtComm":<20} {tc_math_acc:>9.1f}% {tc_math_cons:>9.1f}% {tc_gsm8k_acc:>9.1f}% {tc_gsm8k_cons:>9.1f}%')
print('=' * 85)

# Paper reference
print(f'\nPaper (Qwen3-0.6B):')
print(f'  Single: MATH 45.80%, GSM8K 58.20%')
print(f'  MAF:    MATH 71.20%, GSM8K 70.80%')
print(f'  TC:     MATH 85.00%, GSM8K 75.80%')

# Delta
print(f'\nThoughtComm vs Debate:')
print(f'  MATH: {tc_math_acc - debate_math_acc:+.1f}%')
print(f'  GSM8K: {tc_gsm8k_acc - debate_gsm8k_acc:+.1f}%')

In [ ]:
# Save results
import json

results = {
    'model': config.model_name,
    'version': 'v2',
    'num_eval': config.num_eval,
    'single_answer': {
        'math_acc': sa_math_acc,
        'gsm8k_acc': sa_gsm8k_acc,
    },
    'debate_only': {
        'math_acc_per_agent': debate_math_acc_per_agent,
        'math_acc_avg': debate_math_acc,
        'math_consensus': debate_math_cons,
        'gsm8k_acc_per_agent': debate_gsm8k_acc_per_agent,
        'gsm8k_acc_avg': debate_gsm8k_acc,
        'gsm8k_consensus': debate_gsm8k_cons,
    },
    'thoughtcomm': {
        'math_acc_per_agent': tc_math_acc_per_agent,
        'math_acc_avg': tc_math_acc,
        'math_consensus': tc_math_cons,
        'gsm8k_acc_per_agent': tc_gsm8k_acc_per_agent,
        'gsm8k_acc_avg': tc_gsm8k_acc,
        'gsm8k_consensus': tc_gsm8k_cons,
    },
}

os.makedirs('results', exist_ok=True)
with open(f'results/05v2_final_{MODEL_TAG}.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved: results/05v2_final_{MODEL_TAG}.json')